In [17]:
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from matplotlib.ticker import FuncFormatter
from matplotlib.dates import *

In [18]:
from config import DRIVER as driver

In [38]:
def to_list(cursor):
    return list(map(dict, cursor))

def to_data_frame(cursor):
    return pd.DataFrame(to_list(cursor))

def get_transactions(ind, am):
    url = "http://127.0.0.1:28081/get_outs"
    # Define the JSON payload
    payload = {"outputs":[{"amount": am, "index":ind}], "get_txid": True}
    # Define the headers
    headers = {
        "Content-Type": "application/json"
    }

    # Make the POST request
    response = requests.post(url, json=payload, headers=headers)
    return response.json()

First we will get the number of inputs with a mix in equal to 0. Remember this data has been creaed in a local tesnet, so the number with mixin equal to 0 may be elevated to show the privicy risk of using mixin 0. Rememebre that mixin 0 is not recommended for privacy reasons and you can no longer use it in the Monero main network.

In [10]:
with driver.session() as session:
    num_inputs = to_list(session.run(query=  """MATCH (i:Input) RETURN count(i)"""))[0]['count(i)']
    num_inputs_mix_0 = to_list(session.run(query=  """MATCH (i:Input) WHERE i.mixin = '0' RETURN count(i)"""))[0]['count(i)']

print(f"Number of inputs: {num_inputs}")
print(f"\nNumber of inputs with mixin equal to 0: {num_inputs_mix_0}. This inputs do not have the benefits of ring signatures")

Number of inputs: 127

Number of inputs with mixin equal to 0: 126. This inputs do not have the benefits of ring signatures


Our next step is to get all the information of those inputs with mixins equal to 0 to see what information we can get from them.

In [20]:
query = """MATCH (i:Input)-[:TX_INPUT]->(t:Transactions)
            WHERE i.mixin = '0'
            RETURN i, t"""

with driver.session() as session:
    data = to_list(session.run(query=query))
    
    rows = []
    for record in data:
        input_node = record['i']
        transaction_node = record['t']
        row = {
                    'input_mixin': input_node.get('mixin'),
                    'input_anonset': input_node.get('anonset'),
                    'input_id': input_node.get('id'),
                    'input_value': input_node.get('value'),
                    'key_offset': input_node.get('key_offset'),
                    'transaction_fee': transaction_node.get('fee'),
                    'transaction_id': transaction_node.get('id'),
                    'transaction_hash': transaction_node.get('hash')
                }
        rows.append(row)
inputs = pd.DataFrame(rows)
inputs

,input_mixin,input_anonset,input_id,input_value,key_offset,transaction_fee,transaction_id,transaction_hash
0,0,80,i0,500000000000,[1],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
1,0,80,i1,90000000000,[13],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
2,0,80,i2,10000000000000,[12],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
3,0,11,i3,2000000000,[3],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
4,0,80,i4,7000000000000,[6],5257795553,t81,83f48ef7721c1c7d44df7d11c191f5a5f06aab213cd1d6...
...,...,...,...,...,...,...,...,...
121,0,93,i121,90000000000,[27],4556569224,t118,10ca75a51580e233b70aef08df952fb9efbf7e0f22df37...
122,0,108,i122,10000000000000,[28],2394344630,t119,516b7d7bbcb748d23aeea64459db1d688eff59ce783344...
123,0,89,i123,500000000000,[28],2394344630,t119,516b7d7bbcb748d23aeea64459db1d688eff59ce783344...
124,0,93,i124,90000000000,[28],2394344630,t119,516b7d7bbcb748d23aeea64459db1d688eff59ce783344...


Notice that there are key_offset that repeat. This is because the key_offset it point us at the output of some previous block, and that block can have multiple output. Is the key (which is an information will get on the next steps) along with the key_offset that will help us to identify the output that is being spent.

In [61]:
k_offsets_mix_0 = {}
tx_used = []
for _, row in inputs[['input_value', 'key_offset', 'transaction_hash']].iterrows():
    key_offset = row.key_offset.split('[')[1].split(']')[0]
    value = row.input_value
    outs = get_transactions(int(key_offset), int(value))
    if key_offset not in k_offsets_mix_0.keys():
        k_offsets_mix_0[key_offset] = [{'block': outs['outs'][0]['height'], 'key': outs['outs'][0]['key'], 'amount': value }]
    else:
        k_offsets_mix_0[key_offset].append({'block': outs['outs'][0]['height'], 'key': outs['outs'][0]['key'], 'amount': value })
    tx_used.append(outs['outs'][0]['key'])

With this last dictionary, we have the information of each key_offset used in the transactions with mixin equal to 0. As the mixins is 0, the transactions that appears is 100% the one being spend. If this key it is used in another transaction, we will know that it is not the real one as it has been previosuly spent.

In [54]:
query = """MATCH (i:Input)-[:TX_INPUT]->(t:Transactions)
            WHERE i.mixin <> '0'
            RETURN i, t"""

with driver.session() as session:
    data = to_list(session.run(query=query))
    
    rows = []
    for record in data:
        input_node = record['i']
        transaction_node = record['t']
        row = {
                    'input_mixin': input_node.get('mixin'),
                    'input_anonset': input_node.get('anonset'),
                    'input_id': input_node.get('id'),
                    'input_value': input_node.get('value'),
                    'key_offset': input_node.get('key_offset'),
                    'transaction_fee': transaction_node.get('fee'),
                    'transaction_id': transaction_node.get('id'),
                    'transaction_hash': transaction_node.get('hash')
                }
        rows.append(row)
inputs_mul_mix = pd.DataFrame(rows)
inputs_mul_mix

,input_mixin,input_anonset,input_id,input_value,key_offset,transaction_fee,transaction_id,transaction_hash
0,1,122,i126,10000000000000,"[27, 72]",2055711202,t133,2add8157b83055e2f5ad320370f8360f984562aa4c46cf...


In [70]:
k_offsets = {} 
for _, row in inputs_mul_mix[['input_value', 'key_offset', 'transaction_hash']].iterrows():
    key_offset = row.key_offset.split('[')[1].split(']')[0].split(',')
    value = row.input_value
    for key in key_offset:
        key_i = int(key)
        out = get_transactions(key_i, int(value))
        if out['outs'][0]['key'] in tx_used:
            num_mixin = len(key_offset)
            if num_mixin == 2:
                tx_used.append()
            if str(num_mixin) in k_offsets.keys():
                k_offsets[num_mixin] += 1
            else:
                k_offsets[num_mixin] = 1
            print('-----------')
            print('Vul found')

In [71]:
k_offsets

{'27': [{'block': 28,
   'key': '5744c583566064ce3cefd0b6b8d45922e66bea2f1c361ab26c4588b4bc7ee702',
   'amount': '10000000000000'}],
 '72': [{'block': 73,
   'key': '73f9cbbd35207474cb29f148eba66e7c6b0cee654a42e442da795ba0200b2ec9',
   'amount': '10000000000000'}]}

In [74]:
for k_off in k_offsets.keys():
    if k_off in k_offsets_mix_0.keys():
        print(k_offsets_mix_0[k_off])

[{'block': 28, 'key': '85734c35000ccb1c066037bc36176f62a540a09d8249a7b61a14d799e1954f42', 'amount': '7000000000000'}, {'block': 28, 'key': '408ad76aff19df7466246395b01ed17d876146c9087a513db796c3db97c403ee', 'amount': '500000000000'}, {'block': 28, 'key': 'c5b6d018b9bbdc0d6f109ec2ab7d67b7798a9bdc3133d0dbfcb2ecaf9e70f811', 'amount': '90000000000'}]
